In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

In [ ]:
df = pd.read_csv('brestCancer.csv')
df = df.loc[:, ~df.columns.str.contains('unnamed', case=False)]
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

le = LabelEncoder()
y = le.fit_transform(df['diagnosis'])          # B=0, M=1
X = df.drop(columns=['diagnosis'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Classes    :", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
dt = DecisionTreeClassifier(criterion='gini', random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

acc_dt  = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt)
rec_dt  = recall_score(y_test, y_pred_dt)
f1_dt   = f1_score(y_test, y_pred_dt)

print(f"Accuracy : {acc_dt:.4f}")
print(f"Precision: {prec_dt:.4f}")
print(f"Recall   : {rec_dt:.4f}")
print(f"F1-Score : {f1_dt:.4f}")

In [ ]:
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

acc_nb  = accuracy_score(y_test, y_pred_nb)
prec_nb = precision_score(y_test, y_pred_nb)
rec_nb  = recall_score(y_test, y_pred_nb)
f1_nb   = f1_score(y_test, y_pred_nb)

print(f"Accuracy : {acc_nb:.4f}")
print(f"Precision: {prec_nb:.4f}")
print(f"Recall   : {rec_nb:.4f}")
print(f"F1-Score : {f1_nb:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, y_pred, title, cmap in zip(
    axes,
    [y_pred_dt, y_pred_nb],
    ['Decision Tree', 'Naïve Bayes'],
    ['Oranges', 'Blues']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, cbar=False,
                xticklabels=['Benign', 'Malignant'],
                yticklabels=['Benign', 'Malignant'], ax=ax)
    ax.set_title(f'{title} – Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18, 7))
plot_tree(dt, max_depth=3, feature_names=X.columns,
          class_names=['Benign', 'Malignant'], filled=True, rounded=True)
plt.title('Decision Tree – Splits, Thresholds and Leaf Nodes (Depth 3)')
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.Series(dt.feature_importances_, index=X.columns)
top10 = importance.nlargest(10).sort_values()

plt.figure(figsize=(7, 5))
top10.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Decision Tree – Top 10 Feature Importances')
plt.xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

In [ ]:
metrics = pd.DataFrame({
    'Model'    : ['Decision Tree', 'Naïve Bayes'],
    'Accuracy' : [round(acc_dt,  4), round(acc_nb,  4)],
    'Precision': [round(prec_dt, 4), round(prec_nb, 4)],
    'Recall'   : [round(rec_dt,  4), round(rec_nb,  4)],
    'F1-Score' : [round(f1_dt,   4), round(f1_nb,   4)],
})
display(metrics)